# LangSmith Observability & Monitoring

This notebook demonstrates how to use the LangSmith Python SDK to:
1. Connect to your LangSmith project
2. Browse recent traces and inspect agent steps
3. Replay / drill into a specific run by ID
4. Build a monitoring summary (latency, token usage, error rate)
5. Attach human-feedback scores to runs

**Pre-requisites**
- `LANGSMITH_API_KEY` set in `.env` or the cell below
- LangSmith tracing enabled in the backend (`LANGSMITH_TRACING=true`)
- At least a few chat messages sent through the API so traces exist

In [1]:
# ── 1. Environment setup ──────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

# Load .env from repo root (LANGSMITH_API_KEY, LANGSMITH_PROJECT)
load_dotenv(dotenv_path="../.env", override=False)

LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = os.environ.get("LANGSMITH_PROJECT", "aks-chatbot")

assert LANGSMITH_API_KEY, "Set LANGSMITH_API_KEY in your .env file or environment."
print(f"Project: {LANGSMITH_PROJECT}")

AssertionError: Set LANGSMITH_API_KEY in your .env file or environment.

In [ ]:
# ── 2. Connect to LangSmith ───────────────────────────────────────────────────
from langsmith import Client

client = Client(api_key=LANGSMITH_API_KEY)

# Verify connection
projects = list(client.list_projects())
project_names = [p.name for p in projects]
print("Available projects:", project_names)
assert LANGSMITH_PROJECT in project_names, (
    f"Project '{LANGSMITH_PROJECT}' not found. "
    "Have you sent any chat messages with tracing enabled?"
)

In [ ]:
# ── 3. Browse recent traces ───────────────────────────────────────────────────
# Each top-level trace corresponds to one call to stream_agent_response.
# Within each trace you can see all LangGraph nodes (agent, tools, LLM calls).

from datetime import datetime, timedelta, timezone

since = datetime.now(timezone.utc) - timedelta(hours=24)

runs = list(
    client.list_runs(
        project_name=LANGSMITH_PROJECT,
        run_type="chain",          # top-level traces only
        start_time=since,
        limit=20,
    )
)

print(f"Found {len(runs)} top-level runs in the last 24 h\n")

for run in runs[:5]:
    status = run.status or "unknown"
    latency_ms = (
        int((run.end_time - run.start_time).total_seconds() * 1000)
        if run.end_time
        else None
    )
    tokens = (
        run.total_tokens
        if hasattr(run, "total_tokens")
        else (run.usage_metadata or {}).get("total_tokens", "n/a")
    )
    print(
        f"[{run.start_time:%H:%M:%S}] {status:>8}  "
        f"{latency_ms or '?':>6} ms  {tokens} tokens  id={run.id}"
    )

In [ ]:
# ── 4. Replay a specific run — drill into agent steps ─────────────────────────
# Replace the run_id below with any ID printed above, or set it to runs[0].id

if runs:
    RUN_ID = str(runs[0].id)
else:
    RUN_ID = ""  # paste an ID here if the list above is empty

assert RUN_ID, "No runs found. Send a chat message first."

# Fetch all child runs (LLM call, tool calls, node executions)
child_runs = list(client.list_runs(project_name=LANGSMITH_PROJECT, trace_id=RUN_ID))
child_runs.sort(key=lambda r: r.start_time)

print(f"Run {RUN_ID} — {len(child_runs)} child spans:\n")
for child in child_runs:
    indent = "  " if child.parent_run_id else ""
    latency = (
        f"{int((child.end_time - child.start_time).total_seconds() * 1000)} ms"
        if child.end_time
        else "ongoing"
    )
    print(f"{indent}[{child.run_type:>6}] {child.name:<40} {latency}")

# Show the LLM inputs and outputs for the first LLM span
llm_runs = [r for r in child_runs if r.run_type == "llm"]
if llm_runs:
    llm = llm_runs[0]
    print("\n── LLM inputs ──")
    for msg in (llm.inputs or {}).get("messages", [[]])[0]:
        role = msg.get("role", "?")
        content = str(msg.get("content", ""))[:200]
        print(f"  [{role}] {content}")
    print("\n── LLM output ──")
    output = (llm.outputs or {}).get("generations", [[{}]])[0]
    if output:
        print(" ", str(output[0].get("text", output[0]))[:300])

In [ ]:
# ── 5. Monitoring summary — last 100 traces ───────────────────────────────────
import statistics

recent = list(
    client.list_runs(
        project_name=LANGSMITH_PROJECT,
        run_type="chain",
        limit=100,
    )
)

latencies = [
    (r.end_time - r.start_time).total_seconds() * 1000
    for r in recent
    if r.end_time
]
errors = [r for r in recent if r.error]
token_counts = [
    (r.usage_metadata or {}).get("total_tokens", 0)
    for r in recent
    if (r.usage_metadata or {}).get("total_tokens")
]

print(f"Monitoring summary — last {len(recent)} runs")
print("=" * 45)
if latencies:
    print(f"Latency  p50: {statistics.median(latencies):.0f} ms")
    print(f"         p95: {sorted(latencies)[int(len(latencies) * 0.95)]:.0f} ms")
    print(f"         max: {max(latencies):.0f} ms")
print(f"Errors:        {len(errors)} / {len(recent)} ({100 * len(errors) / max(len(recent), 1):.1f} %)")
if token_counts:
    print(f"Avg tokens:    {statistics.mean(token_counts):.0f}")
    print(f"Total tokens:  {sum(token_counts)}")

In [ ]:
# ── 6. Attach human-feedback scores ──────────────────────────────────────────
# This is useful for RLHF pipelines, evaluation baselines, and SLO monitoring.
# In production the thumbs-up/down buttons in the chat UI would call this API.

if runs:
    target_run_id = str(runs[0].id)

    feedback = client.create_feedback(
        run_id=target_run_id,
        key="thumbs",           # arbitrary metric name
        score=1,                # 1 = positive, 0 = negative
        comment="Notebook demo feedback — looks good!",
    )
    print(f"Feedback created: id={feedback.id}  run={target_run_id}")
    print(f"View run: https://smith.langchain.com/o/<org>/projects/p/{LANGSMITH_PROJECT}/runs/{target_run_id}")
else:
    print("No runs to attach feedback to.")